In [2]:
import torch
from rdkit import Chem

smiles = 'CCO'
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol) 

# Node Features
atom_features_list = []
for atom in mol.GetAtoms():
    atom_features_list.append([atom.GetAtomicNum()]) # Atomic number
    # Could add more: atom.GetFormalCharge(), atom.GetChiralTag(), atom.GetHybridization()

node_features = torch.tensor(atom_features_list, dtype=torch.float)
print(f"Ethanol has {mol.GetNumAtoms()} atoms.")
print("Node features (Atomic Numbers):\n", node_features)

# Edge Index (Connectivity)
adj = Chem.GetAdjacencyMatrix(mol)
edge_list = []
for i in range(mol.GetNumAtoms()):
    for j in range(i + 1, mol.GetNumAtoms()): # Avoid self-loops and duplicate edges for undirected
        if adj[i, j] == 1:
            edge_list.append((i, j)) # Edge from i to j
            edge_list.append((j, i)) # Edge from j to i (for undirected)

if not edge_list: # Handle case with no bonds (e.g. single atom)
    edge_index = torch.empty((2,0), dtype=torch.long)
else:
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
print("Edge Index (Connectivity):\n", edge_index)

# --- Edge Features ---
# We'll use bond type as a simple feature.
edge_features_list = []
unique_bonds_for_features = {} # To avoid double-counting features for undirected edges
for i in range(edge_index.shape[1]):
    start_node, end_node = edge_index[0, i].item(), edge_index[1, i].item()
    bond = mol.GetBondBetweenAtoms(start_node, end_node)
    if bond:
        # Example: BondType.SINGLE=1.0, BondType.DOUBLE=2.0, etc.
        bt = bond.GetBondTypeAsDouble()
        edge_features_list.append([bt])
        # Could add more: bond.GetIsConjugated(), bond.GetIsAromatic(), bond.IsInRing()

if not edge_features_list:
    edge_features = torch.empty((0,1), dtype=torch.float) # Assuming 1 feature if no edges
else:
    edge_features = torch.tensor(edge_features_list, dtype=torch.float)
print("Edge Features (Bond Types):\n", edge_features)

Ethanol has 9 atoms.
Node features (Atomic Numbers):
 tensor([[6.],
        [6.],
        [8.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.]])
Edge Index (Connectivity):
 tensor([[0, 1, 0, 3, 0, 4, 0, 5, 1, 2, 1, 6, 1, 7, 2, 8],
        [1, 0, 3, 0, 4, 0, 5, 0, 2, 1, 6, 1, 7, 1, 8, 2]])
Edge Features (Bond Types):
 tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.]])


In [6]:
import torch
import torch.nn as nn
from torch_geometric.nn import MessagePassing

class MPNNLayer(MessagePassing):
    def __init__(self, node_feat_dim, edge_feat_dim, message_dim, hidden_dim):
        super(MPNNLayer, self).__init__(aggr='add') # Sum aggregation
        # Message function
        self.message_network = nn.Sequential(
            nn.Linear(node_feat_dim + edge_feat_dim, message_dim),
            nn.ReLU()
        )
        # Update function
        self.update_network = nn.Sequential(
            nn.Linear(node_feat_dim + message_dim, hidden_dim),
            nn.ReLU()
        )

    def forward(self, x, edge_index, edge_attr):
        # x: Node features [num_nodes, node_feat_dim]
        # edge_index: [2, num_edges]
        # edge_attr: [num_edges, edge_feat_dim]
        
        # self.propagate will call message(), aggregate(), and update()
        # We pass x (node features of all nodes) and edge_attr
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr):
        # x_i: features of target nodes [num_edges, node_feat_dim]
        # x_j: features of source nodes [num_edges, node_feat_dim]
        # edge_attr: features of edges [num_edges, edge_feat_dim]
        
        # In MPNN, message M(h_u, h_v, e_uv).
        # PyG's message function is typically defined from the perspective of the edge,
        # so x_j is the source and x_i is the target.
        # For a simpler message M(h_u, e_uv):
        tmp = torch.cat([x_j, edge_attr], dim=1) # Concatenate source node features and edge features
        return self.message_network(tmp)

    def update(self, aggregated_messages, x):
        # aggregated_messages: Sum of messages for each node
        # Concatenate node's own features with aggregated messages
        update_input = torch.cat([x, aggregated_messages], dim=1)
        return self.update_network(update_input)

# Node features: 3 nodes, 2 features each
x_initial = torch.tensor([[1.0, 0.5], [0.8, 0.2], [0.3, 0.9]], dtype=torch.float)
node_feat_dim = x_initial.shape[1]

# Edges: (0,1), (1,0), (0,2), (2,0)
edge_index_example = torch.tensor([[0, 1, 0, 2],
                                   [1, 0, 2, 0]], dtype=torch.long)

# Edge features: 4 edges, 1 feature each (e.g., bond type)
edge_attr_example = torch.tensor([[1.0], [1.0], [2.0], [2.0]], dtype=torch.float)
edge_feat_dim = edge_attr_example.shape[1]

message_compute_dim = 16 # Dimension of computed messages
hidden_out_dim = 8    # Output dimension for node features after update
mpnn_layer = MPNNLayer(node_feat_dim, edge_feat_dim, message_compute_dim, hidden_out_dim)
x_updated = mpnn_layer(x_initial, edge_index_example, edge_attr_example)

print("Initial node features (x):\n", x_initial)
print(f"Node features after one MPNN layer (shape: {x_updated.shape}):\n", x_updated)

Initial node features (x):
 tensor([[1.0000, 0.5000],
        [0.8000, 0.2000],
        [0.3000, 0.9000]])
Node features after one MPNN layer (shape: torch.Size([3, 8])):
 tensor([[0.6237, 0.3848, 0.3994, 1.3598, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3274, 0.2963, 0.2558, 0.4116, 0.0000, 0.0000, 0.0000, 0.1622],
        [0.5797, 0.2039, 0.2609, 0.7201, 0.0000, 0.0000, 0.0000, 0.0000]],
       grad_fn=<ReluBackward0>)


In [ ]:
import torch
# Ensure you have PyTorch Geometric installed
from torch_geometric.nn import global_add_pool, global_mean_pool, global_max_pool

# Example: Node features for a batch of 3 graphs
# Graph 0: 2 nodes, Graph 1: 3 nodes, Graph 2: 1 node
# Total nodes = 2 + 3 + 1 = 6
# Node feature dimension = 4
x = torch.tensor([
    [1, 2, 3, 4], 
    [5, 6, 7, 8],     # Nodes for graph 0
    [9, 10, 11, 12],
    [13, 14, 15, 16], 
    [17, 18, 19, 20], # Nodes for graph 1
    [21, 22, 23, 24]  # Node for graph 2
], dtype=torch.float)

# Batch index tensor: indicates which graph each node belongs to
# In PyTorch Geometric, this tensor assigns a graph index to each node.
# Nodes of graph 0 get index 0, nodes of graph 1 get index 1, etc.
batch_idx = torch.tensor([0, 0, 1, 1, 1, 2]) 

# Global Add Pooling (Sum Pooling)
graph_features_sum = global_add_pool(x, batch_idx)

# Global Mean Pooling
graph_features_mean = global_mean_pool(x, batch_idx)

# Global Max Pooling
graph_features_max = global_max_pool(x, batch_idx)

print("Node features (x):\n", x)
print("Batch index (batch_idx):\n", batch_idx)
print("\nGlobal Add Pooled Features (PyG):\n", graph_features_sum)
print("Global Mean Pooled Features (PyG):\n", graph_features_mean)
print("Global Max Pooled Features (PyG):\n", graph_features_max)

Node features (x):
 tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.],
        [13., 14., 15., 16.],
        [17., 18., 19., 20.],
        [21., 22., 23., 24.]])
Batch index (batch_idx):
 tensor([0, 0, 1, 1, 1, 2])

Global Add Pooled Features (PyG):
 tensor([[ 6.,  8., 10., 12.],
        [39., 42., 45., 48.],
        [21., 22., 23., 24.]])
Global Mean Pooled Features (PyG):
 tensor([[ 3.,  4.,  5.,  6.],
        [13., 14., 15., 16.],
        [21., 22., 23., 24.]])
Global Max Pooled Features (PyG):
 tensor([[ 5.,  6.,  7.,  8.],
        [17., 18., 19., 20.],
        [21., 22., 23., 24.]])


In [ ]:
import torch
from torch_geometric.nn import BatchNorm, LayerNorm, GraphNorm
from torch_geometric.data import Data, Batch # For creating a batch of graphs

feature_dim = 16
# Graph 1
x1 = torch.randn(4, feature_dim)  # 4 nodes, 16 features
data1 = Data(x=x1)
# Graph 2 (different number of nodes)
x2 = torch.randn(3, feature_dim)  # 3 nodes, 16 features
data2 = Data(x=x2)

batch_data = Batch.from_data_list([data1, data2])
node_features = batch_data.x  # Shape: [7, 16]
batch_idx = batch_data.batch # Shape: [7], indicating which graph each node belongs to

print("Original Node Features (first 5 of 7 total nodes):\n", node_features[:5])
print("Batch Index (batch_idx):\n", batch_idx)
print("-" * 40)

# --- 1. BatchNorm Example ---
# BatchNorm normalizes features across all nodes in the batch.
bn_layer = BatchNorm(in_channels=feature_dim)
bn_layer.train() # Set to training mode to update running statistics

normalized_features_bn = bn_layer(node_features)
print("BatchNorm Output (first 5 nodes):\n", normalized_features_bn[:5])
print(f"  Mean (feature 0, BN): {normalized_features_bn[:, 0].mean():.4f}") # Should be close to 0
print(f"  Std (feature 0, BN): {normalized_features_bn[:, 0].std():.4f}")   # Should be close to 1
print("-" * 40)

# --- 2. LayerNorm Example ---
# LayerNorm normalizes features for each node independently across its feature dimension.
ln_layer = LayerNorm(in_channels=feature_dim)
ln_layer.train() # train() call mainly for consistency, LN typically doesn't have running stats

normalized_features_ln = ln_layer(node_features)
print("LayerNorm Output (first 5 nodes):\n", normalized_features_ln[:5])
print(f"  Mean (node 0, LN): {normalized_features_ln[0, :].mean():.4f}") # Should be close to 0
print(f"  Std (node 0, LN): {normalized_features_ln[0, :].std():.4f}")   # Should be close to 1
print("-" * 40)

# --- 3. GraphNorm Example ---
# GraphNorm normalizes features per graph within the batch.
gnorm_layer = GraphNorm(in_channels=feature_dim)
gnorm_layer.train() # train() call mainly for consistency

normalized_features_gnorm = gnorm_layer(node_features, batch_idx) # Requires batch_idx
print("GraphNorm Output (first 5 nodes):\n", normalized_features_gnorm[:5])

# Check mean and std for the first feature within the first graph (graph 0)
nodes_in_graph0_mask = (batch_idx == 0)
print(f"  Mean (feature 0, Graph 0, GraphNorm): {normalized_features_gnorm[nodes_in_graph0_mask, 0].mean():.4f}")
print(f"  Std (feature 0, Graph 0, GraphNorm): {normalized_features_gnorm[nodes_in_graph0_mask, 0].std():.4f}")

# Check mean and std for the first feature within the second graph (graph 1)
nodes_in_graph1_mask = (batch_idx == 1)
print(f"  Mean (feature 0, Graph 1, GraphNorm): {normalized_features_gnorm[nodes_in_graph1_mask, 0].mean():.4f}")
print(f"  Std (feature 0, Graph 1, GraphNorm): {normalized_features_gnorm[nodes_in_graph1_mask, 0].std():.4f}")
print("-" * 40)

Original Node Features (first 5 of 7 total nodes):
 tensor([[-0.8078,  0.3080,  1.8256, -1.3844,  0.4521, -2.1199, -1.4604, -0.8643,
          0.5335, -0.4255, -0.1537,  0.0473,  1.2890, -0.5796, -0.7643, -0.2599],
        [-1.1448,  0.9293,  1.4614,  0.3859, -1.4713, -0.8385, -0.9198,  0.9412,
         -0.6387,  1.4850,  0.2197, -0.7781, -0.7741,  0.6829, -1.8053,  0.7760],
        [ 0.0377, -0.3828,  0.3650,  0.1137,  0.0795,  0.1943, -0.1097,  0.8802,
         -0.7327,  0.5596, -0.9036, -1.1204, -1.1624, -0.2545,  0.9356,  0.9379],
        [ 0.1036,  0.3281,  0.7157, -0.7353, -0.1681,  0.7145,  0.1637, -1.8954,
         -1.0816, -0.0258, -0.1128, -0.4437, -0.2272,  0.5722, -1.8886, -0.8375],
        [ 0.4106,  1.1781, -1.3864, -0.8014, -1.6703,  0.1800, -1.0922,  0.7230,
          0.9703,  0.1236, -0.1009, -0.5441, -0.9741, -2.2007,  0.0912,  0.7836]])
Batch Index (batch_idx):
 tensor([0, 0, 0, 0, 1, 1, 1])
----------------------------------------
BatchNorm Output (first 5 nodes):
 

In [10]:
import torch
from torch_geometric.nn import global_add_pool, global_mean_pool, global_max_pool

# Example: Batch of 2 graphs
node_embeddings_batch = torch.randn(5, 8) # 3+2=5 total nodes, 8 features each
batch_idx = torch.tensor([0, 0, 0, 1, 1])

# --- Readout using Global Add Pooling (Sum) ---
graph_embedding_add = global_add_pool(node_embeddings_batch, batch_idx)
print("Readout with Global Add Pooling:\n", graph_embedding_add)
print("Shape:", graph_embedding_add.shape) # [num_graphs, feature_dim] -> [2, 8]
print("-" * 40)

# --- Readout using Global Mean Pooling ---
graph_embedding_mean = global_mean_pool(node_embeddings_batch, batch_idx)
print("Readout with Global Mean Pooling:\n", graph_embedding_mean)
print("Shape:", graph_embedding_mean.shape) # [2, 8]
print("-" * 40)

# --- Readout using Global Max Pooling ---
graph_embedding_max = global_max_pool(node_embeddings_batch, batch_idx)
print("Readout with Global Max Pooling:\n", graph_embedding_max)
print("Shape:", graph_embedding_max.shape) # [2, 8]
print("-" * 40)

# --- Concatenating multiple readouts ---
# It's common to concatenate features from different pooling methods
concatenated_readout = torch.cat([graph_embedding_mean, graph_embedding_max], dim=1)
print("Concatenated Readout (Mean + Max):\n", concatenated_readout)
print("Shape:", concatenated_readout.shape) # [num_graphs, 2 * feature_dim] -> [2, 16]
print("-" * 40)

# These graph embeddings can now be fed into a classifier or regressor
# For example, for a 2-class classification problem:
num_graphs = graph_embedding_mean.shape[0]
output_dim_after_pooling = graph_embedding_mean.shape[1] # 8 in this case
classifier_head = torch.nn.Linear(output_dim_after_pooling, 2) # Predict 2 classes

graph_predictions = classifier_head(graph_embedding_mean)
print("Example Graph Predictions (using mean pooling):\n", graph_predictions)
print("Shape:", graph_predictions.shape) # [num_graphs, num_classes] -> [2, 2]

Readout with Global Add Pooling:
 tensor([[-2.4205, -2.4161,  0.5749,  0.4177, -0.2331,  1.6708, -3.2371, -0.8490],
        [-3.3880,  0.1261, -1.8699,  1.0665, -2.8714,  1.1212, -1.3959, -0.7777]])
Shape: torch.Size([2, 8])
----------------------------------------
Readout with Global Mean Pooling:
 tensor([[-0.8068, -0.8054,  0.1916,  0.1392, -0.0777,  0.5569, -1.0790, -0.2830],
        [-1.6940,  0.0631, -0.9349,  0.5333, -1.4357,  0.5606, -0.6979, -0.3888]])
Shape: torch.Size([2, 8])
----------------------------------------
Readout with Global Max Pooling:
 tensor([[ 0.5462, -0.0291,  0.8946,  1.8801,  0.2696,  1.6256,  0.7329,  1.7794],
        [-0.9827,  0.4898,  0.3106,  1.0972, -0.0076,  0.9826, -0.1195,  0.0591]])
Shape: torch.Size([2, 8])
----------------------------------------
Concatenated Readout (Mean + Max):
 tensor([[-0.8068, -0.8054,  0.1916,  0.1392, -0.0777,  0.5569, -1.0790, -0.2830,
          0.5462, -0.0291,  0.8946,  1.8801,  0.2696,  1.6256,  0.7329,  1.7794],
  

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, GINConv, global_mean_pool
from torch_geometric.data import Data

# --- Unified GNN Model ---
class GNNModel(nn.Module):
    def __init__(self, conv_layer, in_channels, hidden_channels, out_channels, **kwargs):
        """
        Parameters:
        - conv_layer: The convolutional layer class (e.g., GCNConv, SAGEConv, etc.)
        - in_channels: Input feature dimension
        - hidden_channels: Hidden layer dimension
        - out_channels: Output feature dimension (number of classes)
        - kwargs: Additional arguments for the convolutional layer
        """
        super().__init__()
        self.conv1 = conv_layer(in_channels, hidden_channels, **kwargs)
        self.conv2 = conv_layer(hidden_channels, hidden_channels, **kwargs)
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch_idx):
        # GNN Layers
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        # Readout (Graph-level pooling)
        x_graph = global_mean_pool(x, batch_idx)
        # Classifier
        out = self.classifier(x_graph)
        return F.log_softmax(out, dim=-1)

num_nodes = 6
node_features = 16
num_classes = 3
hidden_channels = 32
out_channels = num_classes
edge_index = torch.tensor([[0, 1, 2, 3, 4, 5],
                           [1, 2, 3, 4, 5, 0]], dtype=torch.long)  # Simple circular graph
x = torch.randn(num_nodes, node_features)
batch_idx = torch.tensor([0, 0, 0, 1, 1, 1])  # Batch assignment for 2 graphs
data = Data(x=x, edge_index=edge_index, batch=batch_idx)

# Instantiate Different Models
gcn_model = GNNModel(GCNConv, node_features, hidden_channels, out_channels)
sage_model = GNNModel(SAGEConv, node_features, hidden_channels, out_channels)
gat_model = GNNModel(lambda in_c, out_c, **kwargs: GATConv(in_c, out_c // 2, heads=2, **kwargs), 
                     node_features, hidden_channels, out_channels)
gin_model = GNNModel(lambda in_c, out_c, **kwargs: GINConv(nn.Sequential(
    nn.Linear(in_c, hidden_channels),
    nn.ReLU(),
    nn.Linear(hidden_channels, out_c)
)), node_features, hidden_channels, out_channels)

# --- Forward Pass ---
print("GCN Output:", gcn_model(data.x, data.edge_index, data.batch))
print("GraphSAGE Output:", sage_model(data.x, data.edge_index, data.batch))
print("GAT Output:", gat_model(data.x, data.edge_index, data.batch))
print("GIN Output:", gin_model(data.x, data.edge_index, data.batch))

GCN Output: tensor([[-1.1247, -1.1310, -1.0426],
        [-1.1552, -1.0938, -1.0496]], grad_fn=<LogSoftmaxBackward0>)
GraphSAGE Output: tensor([[-1.0139, -1.1364, -1.1513],
        [-1.0125, -1.1762, -1.1140]], grad_fn=<LogSoftmaxBackward0>)
GAT Output: tensor([[-1.3004, -0.9950, -1.0277],
        [-1.3014, -0.9552, -1.0698]], grad_fn=<LogSoftmaxBackward0>)
GIN Output: tensor([[-1.0967, -1.0606, -1.1401],
        [-1.1055, -1.0585, -1.1333]], grad_fn=<LogSoftmaxBackward0>)
